<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum4/BD_P04_2411532005_Teguh_Esa_Maulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

K-1. Persiapan Lingkungan dan Pembacaan Dataset

Langkah ini untuk menyiapkan lingkungan kerja Praktikum 4 dengan memasang library yang diperlukan, menghubungkan Google Drive, menentukan direktori data dan penyimpanan, serta memastikan dataset trips_bersih hasil Praktikum 3 tersedia. Dataset Parquet kemudian dibaca menggunakan DuckDB, nama beberapa kolom disederhanakan agar lebih mudah digunakan dalam analisis, dan jumlah baris serta kolom diperiksa sebagai keluaran utama K-1. Selain itu, disiapkan fungsi pengukuran waktu dan sample maksimal 300.000 baris untuk kebutuhan pengujian pada K-2 dan K-3.

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

import os
# hasil Praktikum 3 (K-10)
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd
import duckdb

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Membaca parquet menggunakan DuckDB untuk menghindari masalah unifikasi dictionary PyArrow
def baca_parquet_aman(path):
    # Mencari file parquet di dalam folder target
    target_path = os.path.join(path, "**", "*.parquet") if os.path.isdir(path) else path
    con = duckdb.connect()
    return con.execute(f"SELECT * FROM read_parquet('{target_path}')").df()

df = ukur("baca trips_bersih", lambda: baca_parquet_aman(TRIPS_BERSIH))

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.8 MB/s eta 0:00:00
Mounted at /content/drive


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[baca trips_bersih] 15.5697 s
baris: 2,986,910 | kolom: 18


K-2. Perbandingan Model Relasional dan Key-Value

Langkah ini untuk membandingkan penyimpanan relasional menggunakan SQLite dengan model key-value menggunakan dictionary Python. Pengujian dilakukan melalui pencarian 200 perjalanan dengan tiga metode, yaitu tabel SQL tanpa indeks, tabel SQL dengan indeks pada id_perjalanan, dan key-value berdasarkan kunci perjalanan. Selanjutnya dilakukan perhitungan jumlah perjalanan dan rata-rata total_bayar per wilayah menggunakan SQL GROUP BY dan key-value dengan membuka seluruh nilai JSON. Waktu eksekusi dan kesamaan hasil digunakan untuk membandingkan performa dan ketepatan kedua model.

In [2]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 key-value: kamus (dict) dengan kunci "perjalanan: <id>", nilainya teks JSON
def bangun_kv():
    global kv
    kv = {f"perjalanan: {r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}
    return kv

ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan: {i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan (id_perjalanan)") # buat indeks
r2 = ukur("cari 200 perjalanan SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks) # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 1.9915 s
[key-value: bangun dict] 9.1787 s
jumlah key: 300000
[cari 200 perjalanan SQL tanpa indeks] 7.5443 s
[cari 200 perjalanan SQL dengan indeks] 0.0036 s
[cari 200 perjalanan key-value] 0.0012 s
hasil sama? True
[hitung per wilayah SQL GROUP BY] 0.2104 s
[hitung per wilayah key-value (buka semua nilai)] 1.5151 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


Latihan 1 - Ulangi pencarian dengan 2.000 ID acak

In [3]:
random.seed(42)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_indeks_2000():
    cur = con.cursor()
    return [
        cur.execute(
            "SELECT * FROM perjalanan WHERE id_perjalanan = ?",
            (i,)
        ).fetchone()
        for i in id_uji_2000
    ]

def cari_kv_2000():
    return [
        json.loads(kv[f"perjalanan: {i}"])
        for i in id_uji_2000
    ]

t0 = time.perf_counter()
r_sql_2000 = cari_sql_indeks_2000()
waktu_sql_2000 = time.perf_counter() - t0

t0 = time.perf_counter()
r_kv_2000 = cari_kv_2000()
waktu_kv_2000 = time.perf_counter() - t0

rata_sql_us = waktu_sql_2000 / 2000 * 1_000_000
rata_kv_us = waktu_kv_2000 / 2000 * 1_000_000

print(f"SQL berindeks  : {waktu_sql_2000:.6f} s | rata-rata: {rata_sql_us:.2f} µs/pencarian")
print(f"Key-value      : {waktu_kv_2000:.6f} s | rata-rata: {rata_kv_us:.2f} µs/pencarian")
print("hasil sama?", r_sql_2000[0][0] == r_kv_2000[0]["id_perjalanan"])

SQL berindeks  : 0.034836 s | rata-rata: 17.42 µs/pencarian
Key-value      : 0.010706 s | rata-rata: 5.35 µs/pencarian
hasil sama? True


K-3. Implementasi Document-Oriented dengan TinyDB

Langkah ini untuk menerapkan model document-oriented menggunakan TinyDB dengan struktur nested. Sebanyak 20.000 data perjalanan disimpan sebagai dokumen dengan informasi penjemputan, tujuan, dan biaya yang dikelompokkan dalam objek bertingkat. Query dilakukan untuk mencari perjalanan di Manhattan dengan total pembayaran lebih dari 50 serta menghitung dokumen yang memiliki dan tidak memiliki field tip. Selanjutnya ditambahkan satu dokumen baru dengan field cuaca untuk menunjukkan fleksibilitas document-oriented database dalam menambahkan atribut baru tanpa mengubah struktur dokumen lainnya.



In [4]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar
        },
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}


In [5]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen tanpa mengubah struktur apa pun
tabel.insert({
    "id_perjalanan": 999_999,
    "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"},
    "tujuan": {"zona": 236},
    "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})

print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip     : 15906
dokumen yang TIDAK punya tip     : 4094
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


Latihan2 - Hitung dokumen dengan zona tujuan tertentu dan metode_bayar Tunai


In [6]:
zona_uji = int(sample.head(20_000)["zona_tujuan"].mode().iloc[0])

jumlah_tinydb = len(
    tabel.search(
        (Q.tujuan.zona == zona_uji) &
        (Q.biaya.metode_bayar == "Tunai")
    )
)

jumlah_sql = con.execute(
    """
    SELECT COUNT(*)
    FROM perjalanan
    WHERE zona_tujuan = ?
      AND metode_bayar = 'Tunai'
    """,
    (zona_uji,)
).fetchone()[0]

print("zona tujuan yang diuji:", zona_uji)
print("TinyDB K-3 :", jumlah_tinydb)
print("SQL K-2    :", jumlah_sql)
print("hasil sama?:", jumlah_tinydb == jumlah_sql)

zona tujuan yang diuji: 236
TinyDB K-3 : 155
SQL K-2    : 2074
hasil sama?: False


K-4. Perbandingan DuckDB dan SQLite

Langkah ini untuk membandingkan performa DuckDB dan SQLite dalam melakukan analisis data perjalanan. DuckDB membaca data langsung dari Parquet menggunakan VIEW, sedangkan SQLite menyimpan data dalam tabel per baris sebagai pembanding. Kedua engine digunakan untuk menghitung jumlah perjalanan dan rata-rata total_bayar per wilayah, kemudian hasil dan waktu eksekusinya dibandingkan. Pengujian tambahan dilakukan menggunakan filter wilayah Staten Island untuk menunjukkan manfaat partition pruning pada Parquet.



In [7]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""

hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")

ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())

print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

[DuckDB hitung per wilayah (langsung dari Parquet)] 0.2745 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite muat seluruh baris ke tabel] 14.9340 s
[SQLite hitung per wilayah] 1.7315 s
hasil sama? True


In [8]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB hanya wilayah Staten Island] 0.0429 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


Latihan3 - Membandingkan SUM satu kolom dengan beberapa kolom

In [9]:
SQL_SUM_SATU = """
SELECT SUM(total_bayar) AS total_bayar
FROM perjalanan
"""

SQL_SUM_BANYAK = """
SELECT
    SUM(total_bayar) AS total_bayar,
    SUM(jarak_mil) AS total_jarak,
    SUM(zona_jemput) AS total_zona_jemput,
    SUM(zona_tujuan) AS total_zona_tujuan
FROM perjalanan
"""

t0 = time.perf_counter()
sum_satu = con_d.sql(SQL_SUM_SATU).fetchall()
waktu_satu = time.perf_counter() - t0

t0 = time.perf_counter()
sum_banyak = con_d.sql(SQL_SUM_BANYAK).fetchall()
waktu_banyak = time.perf_counter() - t0

print("SUM satu kolom   :", sum_satu)
print(f"waktu             : {waktu_satu:.6f} s")

print("SUM beberapa kolom:", sum_banyak)
print(f"waktu             : {waktu_banyak:.6f} s")

print(f"perbandingan waktu: {waktu_banyak / waktu_satu:.2f}x")

SUM satu kolom   : [(81621442.45996799,)]
waktu             : 0.197834 s
SUM beberapa kolom: [(81621442.45996696, 10231889.610000137, 497452755, 491330400)]
waktu             : 0.358656 s
perbandingan waktu: 1.81x


K-5. Analisis Graph Arus Perjalanan Antar Zona

Langkah ini untuk memodelkan hubungan perjalanan antar zona menggunakan graph berarah dengan NetworkX. Setiap zona direpresentasikan sebagai node, sedangkan arus perjalanan dari zona asal ke zona tujuan menjadi edge dengan bobot berupa jumlah perjalanan. Lima arus terbesar dibandingkan dengan hasil SQL untuk memastikan kesamaan hasil. Selanjutnya dilakukan pencarian zona yang dapat dicapai dalam dua langkah menggunakan traversal graph dan SQL self-join, kemudian dihitung jumlah zona yang dapat dijangkau dalam paling banyak 1, 2, 3, dan 4 langkah.


In [10]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
FROM perjalanan GROUP BY 1, 2""").df())

# Zona node (titik), arus edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))

print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.3141 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


In [11]:
MIN = 50              # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?

# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
SELECT DISTINCT a2.tujuan
FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
  AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah graph (NetworkX)] 0.0066 s
[2 langkah SQL self-join (DuckDB)] 0.0124 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


Latihan4 - Lima zona dengan jumlah tujuan berbeda terbanyak


In [12]:
top_out_graph = sorted(
    G.out_degree(),
    key=lambda x: (-x[1], x[0])
)[:5]

top_out_sql = con_d.sql("""
SELECT
    zona_jemput AS asal,
    COUNT(DISTINCT zona_tujuan) AS jumlah_tujuan_berbeda
FROM perjalanan
GROUP BY zona_jemput
ORDER BY jumlah_tujuan_berbeda DESC, asal
LIMIT 5
""").fetchall()

print("Graph (out-degree):")
print(top_out_graph)

print("\nSQL (COUNT DISTINCT):")
print(top_out_sql)

graph_dict = dict(top_out_graph)
sql_dict = {int(asal): int(jumlah) for asal, jumlah in top_out_sql}

print("\nhasil sama?:", graph_dict == sql_dict)

Graph (out-degree):
[(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]

SQL (COUNT DISTINCT):
[(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]

hasil sama?: True


K-6. Pencarian Spasial dengan STRtree

Langkah ini untuk membandingkan beberapa metode pencarian spasial menggunakan data titik sintetis karena dataset TLC hanya menyediakan ID zona tanpa koordinat. Sebanyak 200.000 titik dan 1.000 wilayah pencarian berbentuk kotak dibuat untuk pengujian. Pencarian dilakukan menggunakan loop Python, operasi vectorized NumPy, dan indeks spasial STRtree. Waktu eksekusi dan kesamaan hasil dibandingkan untuk mengetahui metode yang lebih efisien. STRtree digunakan sebagai indeks spasial agar pencarian titik pada banyak wilayah dapat dilakukan lebih cepat dibandingkan pemeriksaan seluruh titik secara langsung.

In [13]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
lintang = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70) # longitude
bujur = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)      # latitude
titik = points(lintang, bujur)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (lintang >= x - 0.005) & (lintang <= x + 0.005) & (bujur >= y - 0.005) & (bujur <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 loop Python, 5 kotak] 18.5939 s
[cara 2 NumPy, 1000 kotak] 0.6143 s
[bangun indeks STRtree] 0.0884 s
[cara 3 STRtree, 1000 kotak] 0.2624 s
5 kotak pertama loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


Latihan5 - Perbandingan STRtree dan NumPy untuk ukuran kotak berbeda

In [14]:
def uji_spasial(sisi):
    kotak_uji = [
        box(
            x - sisi / 2, y - sisi / 2,
            x + sisi / 2, y + sisi / 2
        )
        for x, y in zip(cx, cy)
    ]

    t0 = time.perf_counter()
    hasil_np = []

    for x, y in zip(cx, cy):
        m = (
            (lintang >= x - sisi / 2) &
            (lintang <= x + sisi / 2) &
            (bujur >= y - sisi / 2) &
            (bujur <= y + sisi / 2)
        )
        hasil_np.append(int(m.sum()))

    waktu_np = time.perf_counter() - t0

    t0 = time.perf_counter()
    hasil_str = [
        len(pohon.query(k, predicate="contains"))
        for k in kotak_uji
    ]
    waktu_str = time.perf_counter() - t0

    return waktu_np, waktu_str, hasil_np == hasil_str

for sisi in [0.02, 0.001]:
    waktu_np, waktu_str, sama = uji_spasial(sisi)

    print(f"\nSisi kotak: {sisi} derajat")
    print(f"NumPy   : {waktu_np:.6f} s")
    print(f"STRtree : {waktu_str:.6f} s")
    print(f"STRtree/NumPy: {waktu_str / waktu_np:.4f}x")
    print("hasil sama?:", sama)


Sisi kotak: 0.02 derajat
NumPy   : 0.712614 s
STRtree : 0.958551 s
STRtree/NumPy: 1.3451x
hasil sama?: True

Sisi kotak: 0.001 derajat
NumPy   : 0.687821 s
STRtree : 0.018470 s
STRtree/NumPy: 0.0269x
hasil sama?: True


K-7. Delta Lake, Versioning, dan Time Travel

Langkah ini untuk menerapkan Delta Lake sebagai penyimpanan data yang mendukung versioning, time travel, update, dan schema enforcement. Data awal sebanyak 100.000 baris disimpan sebagai versi 0, kemudian 50.000 baris ditambahkan sehingga terbentuk versi 1. Selanjutnya dilakukan koreksi pada data tertentu untuk menghasilkan versi 2 dan dibandingkan dengan versi sebelumnya menggunakan fitur time travel. Riwayat transaksi Delta Lake diperiksa untuk melihat operasi pada setiap versi. Terakhir, dilakukan pengujian schema enforcement dengan menambahkan kolom asing untuk memastikan data dengan struktur yang tidak sesuai ditolak dan versi tabel tidak berubah.



In [15]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

# Ganti ke local path karena Google Drive (fuse) tidak mendukung atomic rename yang diperlukan Delta Lake
PATH_DELTA = "/content/trips_delta"

if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

# Jika ingin mencadangkannya ke Google Drive, kita bisa menyalin hasil folder di akhir langkah
PATH_DELTA_DRIVE = f"{DIR_SIMPAN}/trips_delta"
if os.path.exists(PATH_DELTA_DRIVE):
    shutil.rmtree(PATH_DELTA_DRIVE)
shutil.copytree(PATH_DELTA, PATH_DELTA_DRIVE)

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2


'/content/drive/MyDrive/BigData/Praktikum4/trips_delta'

In [16]:
# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1

try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)

print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


K-8. Compaction dan Pengelolaan File Delta Lake

Langkah ini untuk menguji dampak penambahan data secara bertahap terhadap jumlah file Delta Lake serta menerapkan compaction untuk menggabungkan file-file kecil. Sebanyak 20 kali append dilakukan dengan masing-masing 500 baris, kemudian jumlah file aktif, versi tabel, jumlah baris, dan waktu pembacaan dibandingkan sebelum dan sesudah compaction. Selanjutnya diperiksa jumlah file Parquet yang masih tersimpan secara fisik di disk untuk menunjukkan bahwa file lama tidak langsung dihapus setelah compaction. Fitur VACUUM dijalankan dalam mode dry_run untuk mengetahui jumlah file lama yang menjadi kandidat penghapusan tanpa benar-benar menghapusnya.


In [17]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) # dry_run hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0758 s | sesudah: 0.0542 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


Latihan6 - Time travel dan restore

In [18]:
dt = DeltaTable(PATH_DELTA)

print(
    "versi sebelum restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

dt.restore(1)

dt = DeltaTable(PATH_DELTA)

print(
    "versi setelah restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


K-9. Rekapitulasi Benchmark

Langkah ini untuk mengumpulkan seluruh hasil pengukuran waktu dari tahapan praktikum sebelumnya ke dalam satu tabel benchmark. Data pada variabel catatan dikonversi menjadi DataFrame, kemudian disimpan sebagai benchmark_nosql.csv pada folder Praktikum 4. Tabel rekapitulasi digunakan untuk melihat dan membandingkan waktu eksekusi berbagai metode penyimpanan, query, analisis graph, dan pencarian spasial yang telah diuji.

In [19]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                          langkah   detik
                                baca trips_bersih 15.5697
                           relasional: muat tabel  1.9915
                           key-value: bangun dict  9.1787
             cari 200 perjalanan SQL tanpa indeks  7.5443
            cari 200 perjalanan SQL dengan indeks  0.0036
                    cari 200 perjalanan key-value  0.0012
                  hitung per wilayah SQL GROUP BY  0.2104
  hitung per wilayah key-value (buka semua nilai)  1.5151
DuckDB hitung per wilayah (langsung dari Parquet)  0.2745
               SQLite muat seluruh baris ke tabel 14.9340
                        SQLite hitung per wilayah  1.7315
               DuckDB hanya wilayah Staten Island  0.0429
                           hitung arus antar zona  0.3141
                       2 langkah graph (NetworkX)  0.0066
                 2 langkah SQL self-join (DuckDB)  0.0124
                      cara 1 loop Python, 5 kotak 18.5939
              

R-3. Simulasi Delta Lake Dua Hari

Simulasi ini digunakan untuk menunjukkan penerapan Delta Lake dalam dua hari kerja. Pada hari pertama, sebagian data ditulis ke tabel Delta Lake sebagai kondisi awal. Pada hari kedua, data baru ditambahkan dan beberapa baris dikoreksi. Delta Lake kemudian digunakan untuk menunjukkan kemampuan **time travel**, yaitu membaca kembali kondisi tabel seperti pada akhir hari pertama meskipun tabel telah mengalami perubahan pada hari kedua.

In [20]:
dt = DeltaTable(PATH_DELTA)

hari1 = DeltaTable(PATH_DELTA, version=0).to_pandas()

print("KONDISI AKHIR HARI 1")
print("Versi Delta :", 0)
print("Jumlah baris:", len(hari1))

hari2 = DeltaTable(PATH_DELTA).to_pandas()
versi_terbaru = DeltaTable(PATH_DELTA).version()

print("\n KONDISI AKHIR HARI 2")
print("Versi Delta :", versi_terbaru)
print("Jumlah baris:", len(hari2))

time_travel = DeltaTable(PATH_DELTA, version=0).to_pandas()

print("\nTIME TRAVEL")
print("Berhasil membaca kembali versi Hari 1")
print("Versi yang dibaca :", 0)
print("Jumlah baris      :", len(time_travel))

print("\nRIWAYAT DELTA LAKE")
for h in dt.history():
    print(
        "Versi:", h.get("version"),
        "| Operasi:", h.get("operation")
    )

LOG_PATH = f"{PATH_DELTA}/_delta_log"

print("\nISI _delta_log/")
log_files = sorted(os.listdir(LOG_PATH))

for nama in log_files:
    print(nama)

print("\nJumlah file _delta_log:", len(log_files))

KONDISI AKHIR HARI 1
Versi Delta : 0
Jumlah baris: 100000

 KONDISI AKHIR HARI 2
Versi Delta : 24
Jumlah baris: 150000

TIME TRAVEL
Berhasil membaca kembali versi Hari 1
Versi yang dibaca : 0
Jumlah baris      : 100000

RIWAYAT DELTA LAKE
Versi: 24 | Operasi: RESTORE
Versi: 23 | Operasi: OPTIMIZE
Versi: 22 | Operasi: WRITE
Versi: 21 | Operasi: WRITE
Versi: 20 | Operasi: WRITE
Versi: 19 | Operasi: WRITE
Versi: 18 | Operasi: WRITE
Versi: 17 | Operasi: WRITE
Versi: 16 | Operasi: WRITE
Versi: 15 | Operasi: WRITE
Versi: 14 | Operasi: WRITE
Versi: 13 | Operasi: WRITE
Versi: 12 | Operasi: WRITE
Versi: 11 | Operasi: WRITE
Versi: 10 | Operasi: WRITE
Versi: 9 | Operasi: WRITE
Versi: 8 | Operasi: WRITE
Versi: 7 | Operasi: WRITE
Versi: 6 | Operasi: WRITE
Versi: 5 | Operasi: WRITE
Versi: 4 | Operasi: WRITE
Versi: 3 | Operasi: WRITE
Versi: 2 | Operasi: UPDATE
Versi: 1 | Operasi: WRITE
Versi: 0 | Operasi: WRITE

ISI _delta_log/
00000000000000000000.json
00000000000000000001.json
00000000000000000002.